# 01 — Compute Per-Query Metrics

Turns TREC run files (`*_run.txt`) + qrels into tidy per-query CSVs in `../data/computed/`. These power Figs 4.1, 4.4, 4.12, 4.13, 4.14.

Outputs:
- `per_query_baseline_dense.csv` — qid, ndcg10, recall10, recall100, mrr
- `per_query_baseline_bm25.csv`
- `per_query_csqe_bm25.csv`
- `per_query_csqe_dense.csv`
- `per_query_csqe_hybrid_rrf.csv`
- `query_lengths.csv` — qid, query, word_count, length_bin
- `csqe_vs_blind_delta.csv` — qid, csqe_ndcg, blind_ndcg, delta, length_bin (one row per query)

## Pre-reqs
```
pip install pytrec_eval pandas
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _helpers import *

DATA_COMPUTED.mkdir(parents=True, exist_ok=True)
qrels = load_json(DATA_RAW / 'miracl_qrels_dev.json')
print(f'qrels loaded: {len(qrels)} queries')

In [ ]:
# --- Per-query metrics for each TREC run ---
runs = {
    'baseline_dense':    DATA_RAW / 'baseline_dense_run.txt',
    'baseline_bm25':     DATA_RAW / 'baseline_bm25_run.txt',
    'csqe_bm25':         DATA_RAW / 'bm25_csqe_run.txt',
    'csqe_dense':        DATA_RAW / 'dense_csqe_run.txt',
    'csqe_hybrid_rrf':   DATA_RAW / 'hybrid_csqe_rrf_k20.txt',
}
for name, path in runs.items():
    if not path.exists():
        print(f'  SKIP {name}: missing {path.name}')
        continue
    df = load_trec_run(path)
    metrics = compute_per_query_metrics(df, qrels)
    out = DATA_COMPUTED / f'per_query_{name}.csv'
    metrics.to_csv(out, index=False)
    print(f'  {name}: {len(metrics)} queries -> {out.name}')

In [ ]:
# --- Query lengths from topics ---
topics = load_json(DATA_RAW / 'miracl_topics_dev.json')
rows = []
for qid, q in topics.items():
    text = q if isinstance(q, str) else q.get('query', q.get('text', ''))
    n = len(text.split())
    rows.append({'qid': str(qid), 'query': text, 'word_count': n, 'length_bin': length_bin(n)})
ql = pd.DataFrame(rows)
ql.to_csv(DATA_COMPUTED / 'query_lengths.csv', index=False)
print(f'query_lengths.csv: {len(ql)} queries; bins: {ql.length_bin.value_counts().to_dict()}')

In [ ]:
# --- CSQE vs blind QE delta per query (for Fig 4.12) ---
# Blind QE per-query for Aya isn't in repo by default. If you have it, place at:
#   DATA_RAW / 'per_query_aya_blind_bm25.csv'  with columns qid,ndcg10
# Otherwise this cell falls back to CSQE Hybrid - BM25 baseline delta.
csqe = pd.read_csv(DATA_COMPUTED / 'per_query_csqe_hybrid_rrf.csv', dtype={'qid': str})
blind_path = DATA_RAW / 'per_query_aya_blind_bm25.csv'
if blind_path.exists():
    blind = pd.read_csv(blind_path, dtype={'qid': str})
    label = 'csqe_minus_aya_blind'
else:
    blind = pd.read_csv(DATA_COMPUTED / 'per_query_baseline_bm25.csv', dtype={'qid': str})
    label = 'csqe_minus_bm25_baseline'
    print('  using BM25 baseline as fallback for blind (Aya blind per-query not on disk).')
merged = csqe.merge(blind, on='qid', suffixes=('_csqe', '_blind')).merge(ql[['qid','word_count','length_bin']], on='qid', how='left')
merged['delta'] = merged['ndcg10_csqe'] - merged['ndcg10_blind']
merged[['qid','ndcg10_csqe','ndcg10_blind','delta','word_count','length_bin']].to_csv(DATA_COMPUTED / 'csqe_vs_blind_delta.csv', index=False)
print(f'  csqe_vs_blind_delta.csv: {len(merged)} rows; reference={label}; mean delta={merged.delta.mean():.4f}')